In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType
from pyspark.sql import Row

catalog_name = "ecommerce"

In [0]:
df_products_clean = spark.read.table(f"{catalog_name}.silver.slv_products_clean")
df_brands_clean = spark.read.table(f"{catalog_name}.silver.slv_brands_clean")
df_categories_clean = spark.read.table(f"{catalog_name}.silver.slv_categories_clean")


In [0]:
df_products_clean.createOrReplaceTempView("view_products")
df_brands_clean.createOrReplaceTempView("view_brands")
df_categories_clean.createOrReplaceTempView("view_categories")

In [0]:
display(spark.sql("SELECT * FROM view_products LIMIT 5"))

In [0]:
display(spark.sql("SELECT * FROM view_categories LIMIT 5"))

In [0]:
display(spark.sql("SELECT * FROM view_brands LIMIT 5"))

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
%sql

CREATE OR REPLACE TABLE gold.gld_dim_products AS

WITH brands_categories AS (
  SELECT
    b.brand_name,
    b.brand_code,
    c.category_name,
    c.category_code
  FROM view_brands b
  INNER JOIN view_categories c
  ON b.category_code = c.category_code
)

SELECT
  p.product_id,
  p.sku,
  p.category_code,
  COALESCE(bc.category_name, 'Not Available') AS category_name,
  p.brand_code,
  COALESCE(bc.brand_name, 'Not Available') AS brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating,
  p.rating_count,
  p.file_name,
  p.ingest_timestamp
FROM view_products p
LEFT JOIN brands_categories bc
ON p.brand_code = bc.brand_code;